# Genetic Algorithm Repeated Runs and Feasibility Validation

The instance is generated once using `INSTANCE_SEED`. Each replication changes only `GA_SEED`, so all runs solve exactly the same problem instance. For a different experimental instance, update only the instance-configuration block and retain the replication workflow.


In [86]:
import math
import random
import time
from collections import defaultdict

import pandas as pd

# ============================================================
# INSTANCE CONFIGURATION
# The instance seed is fixed across all GA replications.
# ============================================================
INSTANCE_SEED = 42
instance_rng = random.Random(INSTANCE_SEED)

nurses = list(range(1, 11))
patients = list(range(1, 61))
services = [1, 2, 3, 4, 5]
nodes = [0] + patients
advance_patients = list(range(1, 31))
dynamic_patients = list(range(31, 61))

coords = {0: (25, 25)}
for i in patients:
    coords[i] = (instance_rng.uniform(0, 50), instance_rng.uniform(0, 50))

travel_time = {}
speed = 1.0
for i in nodes:
    for j in nodes:
        if i != j:
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            travel_time[(i, j)] = math.hypot(dx, dy) / speed

#service_duration = {1: 30, 2: 45}
#service_duration = {1: 30, 2: 45, 3: 60, 4:20}
service_duration = {1: 30, 2: 45, 3: 30, 4:20, 5:60}

# Each request node requires exactly one service.
request = {i: [instance_rng.randint(1, 5)] for i in patients}

sync_patients = instance_rng.sample(patients, 12)
sync = {i: int(i in sync_patients) for i in patients}

priority = {i: instance_rng.randint(3, 5) for i in patients}

patient_time_windows = {}
for i in advance_patients:
    start = instance_rng.randrange(0, 301, 15)
    patient_time_windows[i] = (start, start + 180)

nurse_time_windows = {k: (0, 480) for k in nurses}

qual = {(k, s): 0 for k in nurses for s in services}
for s in services:
    assigned_nurses = nurses[(s - 1) * 2:s * 2]
    for k in assigned_nurses:
        qual[(k, s)] = 1

request_time = {
    i: instance_rng.randint(0, 300)
    for i in dynamic_patients
}

def tt(i, j):
    # Zero travel when two consecutive genes use the same location.
    return 0.0 if i == j else travel_time[(i, j)]

# One chromosome position per required caregiver assignment.
gene_map = []
for i in patients:
    s = request[i][0]
    copies = 2 if sync[i] else 1
    for copy_number in range(1, copies + 1):
        gene_map.append((i, s, copy_number))

print(f"Instance seed: {INSTANCE_SEED}")
print(f"Patients: {len(patients)} | Advance: {len(advance_patients)} | Dynamic: {len(dynamic_patients)}")
print(f"Synchronized requests: {sorted(sync_patients)}")


Instance seed: 42
Patients: 60 | Advance: 30 | Dynamic: 30
Synchronized requests: [9, 14, 16, 27, 31, 32, 35, 37, 43, 47, 52, 54]


In [87]:
# ============================================================
# CHROMOSOME GENERATION, DECODING, AND VALIDATION
# ============================================================

def generate_chromosome(rng):
    chromosome = []
    used_sync_nurses = {}

    for i, s, copy_number in gene_map:
        qualified = [k for k in nurses if qual.get((k, s), 0) == 1]

        if sync[i]:
            if len(qualified) < 2:
                raise ValueError(f"Synchronized request {i} has fewer than two qualified caregivers.")
            if (i, s) not in used_sync_nurses:
                used_sync_nurses[(i, s)] = rng.sample(qualified, 2)
            nurse = used_sync_nurses[(i, s)][copy_number - 1]
        else:
            nurse = rng.choice(qualified)

        order = rng.uniform(0, 1)
        chromosome.append(round(nurse + order, 6))

    return chromosome


def decode_chromosome(chromosome):
    chromosome = chromosome[:]

    # Give both copies of a synchronized request the same ordering key.
    sync_indices = {}
    for idx, (i, s, _) in enumerate(gene_map):
        if sync[i]:
            sync_indices.setdefault((i, s), []).append(idx)

    for idxs in sync_indices.values():
        if len(idxs) == 2:
            g1, g2 = chromosome[idxs[0]], chromosome[idxs[1]]
            common_order = min(g1 % 1, g2 % 1)
            chromosome[idxs[0]] = int(g1) + common_order
            chromosome[idxs[1]] = int(g2) + common_order

    nurse_routes = defaultdict(list)
    for idx, (i, s, _) in enumerate(gene_map):
        nurse = int(chromosome[idx])
        if qual.get((nurse, s), 0) == 1:
            nurse_routes[nurse].append((i, chromosome[idx] % 1, s))

    schedule = defaultdict(list)
    uncovered = set()

    # Initial route construction.
    for nurse in nurses:
        route = sorted(nurse_routes[nurse], key=lambda x: x[1])
        current_time = nurse_time_windows[nurse][0]
        location = 0

        for i, _, s in route:
            current_time += tt(location, i)

            if i in advance_patients:
                earliest, latest = patient_time_windows[i]
                if current_time > latest:
                    uncovered.add(i)
                    continue
                current_time = max(current_time, earliest)
            else:
                current_time = max(current_time, request_time[i])

            finish = current_time + service_duration[s]
            if finish + tt(i, 0) > nurse_time_windows[nurse][1]:
                uncovered.add(i)
                continue

            schedule[nurse].append((i, current_time, s))
            current_time = finish
            location = i

    def simulate_route(nurse, forced_starts):
        rebuilt = []
        current_time = nurse_time_windows[nurse][0]
        location = 0

        for patient, _, service in schedule[nurse]:
            arrival = current_time + tt(location, patient)

            if patient in forced_starts:
                start_time = forced_starts[patient]
                if start_time + 1e-8 < arrival:
                    return None
            else:
                start_time = arrival
                if patient in advance_patients:
                    earliest, latest = patient_time_windows[patient]
                    start_time = max(start_time, earliest)
                    if start_time > latest + 1e-8:
                        return None
                else:
                    start_time = max(start_time, request_time[patient])

            finish = start_time + service_duration[service]
            if finish + tt(patient, 0) > nurse_time_windows[nurse][1] + 1e-8:
                return None

            rebuilt.append((patient, start_time, service))
            current_time = finish
            location = patient

        return rebuilt

    def natural_start(nurse, target_patient, forced_starts):
        current_time = nurse_time_windows[nurse][0]
        location = 0

        for patient, _, service in schedule[nurse]:
            arrival = current_time + tt(location, patient)

            if patient == target_patient:
                start_time = arrival
                if patient in advance_patients:
                    earliest, latest = patient_time_windows[patient]
                    start_time = max(start_time, earliest)
                    if start_time > latest + 1e-8:
                        return None
                else:
                    start_time = max(start_time, request_time[patient])
                return start_time

            if patient in forced_starts:
                start_time = forced_starts[patient]
                if start_time + 1e-8 < arrival:
                    return None
            else:
                start_time = arrival
                if patient in advance_patients:
                    earliest, latest = patient_time_windows[patient]
                    start_time = max(start_time, earliest)
                    if start_time > latest + 1e-8:
                        return None
                else:
                    start_time = max(start_time, request_time[patient])

            current_time = start_time + service_duration[service]
            location = patient

        return None

    forced_starts = defaultdict(dict)

    # Synchronization repair/rejection.
    for (i, s), idxs in sync_indices.items():
        n1 = int(chromosome[idxs[0]])
        n2 = int(chromosome[idxs[1]])

        # Crossover or mutation may assign the same caregiver twice.
        if n1 == n2:
            schedule[n1] = [visit for visit in schedule[n1] if visit[0] != i]
            uncovered.add(i)
            continue

        visit1 = any(visit[0] == i for visit in schedule[n1])
        visit2 = any(visit[0] == i for visit in schedule[n2])

        # A synchronized request is all-or-nothing. Remove any partial visit.
        if not (visit1 and visit2):
            schedule[n1] = [visit for visit in schedule[n1] if visit[0] != i]
            schedule[n2] = [visit for visit in schedule[n2] if visit[0] != i]
            uncovered.add(i)
            continue

        arrival1 = natural_start(n1, i, forced_starts[n1])
        arrival2 = natural_start(n2, i, forced_starts[n2])

        if arrival1 is None or arrival2 is None:
            schedule[n1] = [visit for visit in schedule[n1] if visit[0] != i]
            schedule[n2] = [visit for visit in schedule[n2] if visit[0] != i]
            uncovered.add(i)
            continue

        sync_start = max(arrival1, arrival2)
        if i in dynamic_patients:
            sync_start = max(sync_start, request_time[i])

        if i in advance_patients and sync_start > patient_time_windows[i][1] + 1e-8:
            schedule[n1] = [visit for visit in schedule[n1] if visit[0] != i]
            schedule[n2] = [visit for visit in schedule[n2] if visit[0] != i]
            uncovered.add(i)
            continue

        candidate_forced_1 = dict(forced_starts[n1])
        candidate_forced_2 = dict(forced_starts[n2])
        candidate_forced_1[i] = sync_start
        candidate_forced_2[i] = sync_start

        rebuilt1 = simulate_route(n1, candidate_forced_1)
        rebuilt2 = simulate_route(n2, candidate_forced_2)

        if rebuilt1 is None or rebuilt2 is None:
            schedule[n1] = [visit for visit in schedule[n1] if visit[0] != i]
            schedule[n2] = [visit for visit in schedule[n2] if visit[0] != i]
            uncovered.add(i)
            continue

        schedule[n1] = rebuilt1
        schedule[n2] = rebuilt2
        forced_starts[n1] = candidate_forced_1
        forced_starts[n2] = candidate_forced_2
        uncovered.discard(i)

    # A non-synchronized request is covered only if it appears exactly once.
    appearances = defaultdict(int)
    for visits in schedule.values():
        for i, _, _ in visits:
            appearances[i] += 1

    for i in patients:
        required = 2 if sync[i] else 1
        if appearances[i] != required:
            uncovered.add(i)
            for nurse in nurses:
                schedule[nurse] = [visit for visit in schedule[nurse] if visit[0] != i]

    return schedule, uncovered


def validate_schedule(schedule, uncovered, tolerance=1e-6):
    errors = []
    appearances = defaultdict(list)

    for nurse in nurses:
        visits = schedule.get(nurse, [])
        current_time = nurse_time_windows[nurse][0]
        location = 0

        for patient, start_time, service in visits:
            appearances[patient].append((nurse, start_time, service))

            if service not in request[patient]:
                errors.append(f"Patient {patient}: incorrect service {service}.")
            if qual.get((nurse, service), 0) != 1:
                errors.append(f"Nurse {nurse} is not qualified for service {service}.")

            earliest_reachable = current_time + tt(location, patient)
            if start_time + tolerance < earliest_reachable:
                errors.append(f"Nurse {nurse}: patient {patient} starts before the nurse can arrive.")

            if patient in advance_patients:
                earliest, latest = patient_time_windows[patient]
                if start_time < earliest - tolerance or start_time > latest + tolerance:
                    errors.append(f"Patient {patient}: hard time window violated.")
            elif start_time < request_time[patient] - tolerance:
                errors.append(f"Patient {patient}: served before request arrival.")

            current_time = start_time + service_duration[service]
            location = patient

        if visits and current_time + tt(location, 0) > nurse_time_windows[nurse][1] + tolerance:
            errors.append(f"Nurse {nurse}: cannot return to the depot before shift end.")

    for patient in patients:
        visits = appearances[patient]

        if patient in uncovered:
            if visits:
                errors.append(f"Patient {patient}: marked uncovered but remains in a route.")
            continue

        required = 2 if sync[patient] else 1
        if len(visits) != required:
            errors.append(f"Patient {patient}: expected {required} visits, found {len(visits)}.")

        if sync[patient] and len(visits) == 2:
            if visits[0][0] == visits[1][0]:
                errors.append(f"Patient {patient}: same caregiver assigned twice.")
            if abs(visits[0][1] - visits[1][1]) > tolerance:
                errors.append(f"Patient {patient}: synchronized starts are unequal.")

    return errors


def objective_components(schedule, uncovered, cD=1.0, cT=1.0, cU=1000.0):
    delay = 0.0
    travel = 0.0
    counted_dynamic = set()

    for nurse in nurses:
        previous = 0
        for patient, start_time, service in schedule.get(nurse, []):
            travel += tt(previous, patient)
            if patient in dynamic_patients and patient not in counted_dynamic:
                delay += max(0.0, start_time - request_time[patient])
                counted_dynamic.add(patient)
            previous = patient
        if schedule.get(nurse):
            travel += tt(previous, 0)

    uncovered_priority = sum(priority[i] for i in uncovered)
    coverage_penalty = cU * uncovered_priority
    total = cD * delay + cT * travel + coverage_penalty

    return {
        "total_objective": total,
        "delay_component": cD * delay,
        "travel_component": cT * travel,
        "coverage_component": coverage_penalty,
        "uncovered_priority": uncovered_priority,
        "uncovered_total": len(uncovered),
        "uncovered_advance": sum(i in advance_patients for i in uncovered),
        "uncovered_dynamic": sum(i in dynamic_patients for i in uncovered),
    }


In [88]:
# ============================================================
# GA OPERATORS AND ONE-RUN PROCEDURE
# ============================================================

def tournament_selection(population, fitness, number_selected, rng, tournament_size=5):
    selected = []
    for _ in range(number_selected):
        indices = rng.sample(range(len(population)), tournament_size)
        winner = max(indices, key=lambda idx: fitness[idx])
        selected.append(population[winner])
    return selected


def one_point_crossover(parent1, parent2, rng):
    point = rng.randint(1, len(parent1) - 2)
    return (
        parent1[:point] + parent2[point:],
        parent2[:point] + parent1[point:],
    )


def mutate(chromosome, rng, mutation_probability=0.05):
    child = chromosome[:]
    for idx, (_, service, _) in enumerate(gene_map):
        if rng.random() < mutation_probability:
            qualified = [k for k in nurses if qual.get((k, service), 0) == 1]
            nurse = rng.choice(qualified)
            order = rng.uniform(0, 1)
            child[idx] = round(nurse + order, 6)
    return child


NPOP = 300
NGEN = 200
MUTATION_PROBABILITY = 0.05
ELITE_SIZE = 20
TOURNAMENT_SIZE = 5


def evaluate_chromosome(chromosome):
    schedule, uncovered = decode_chromosome(chromosome)
    errors = validate_schedule(schedule, uncovered)

    if errors:
        return float("-inf"), schedule, uncovered, None, errors

    components = objective_components(schedule, uncovered)
    return -components["total_objective"], schedule, uncovered, components, []


def run_ga(ga_seed, verbose=False):
    rng = random.Random(ga_seed)
    start_clock = time.perf_counter()

    population = [generate_chromosome(rng) for _ in range(NPOP)]

    def evaluate_population(pop):
        values = []
        for chromosome in pop:
            fitness_value, _, _, _, _ = evaluate_chromosome(chromosome[:])
            values.append(fitness_value)
        return values

    fitness = evaluate_population(population)
    best_chromosome = None
    best_fitness = float("-inf")

    def update_best(pop, values, incumbent_chromosome, incumbent_fitness):
        idx = max(range(len(values)), key=lambda j: values[j])
        if values[idx] > incumbent_fitness:
            return pop[idx][:], values[idx]
        return incumbent_chromosome, incumbent_fitness

    best_chromosome, best_fitness = update_best(
        population, fitness, best_chromosome, best_fitness
    )

    for generation in range(NGEN):
        elite_indices = sorted(
            range(len(fitness)), key=lambda idx: fitness[idx], reverse=True
        )[:ELITE_SIZE]
        new_population = [population[idx][:] for idx in elite_indices]

        while len(new_population) < NPOP:
            parent1, parent2 = tournament_selection(
                population,
                fitness,
                2,
                rng,
                tournament_size=TOURNAMENT_SIZE,
            )
            child1, child2 = one_point_crossover(parent1, parent2, rng)
            child1 = mutate(child1, rng, MUTATION_PROBABILITY)
            child2 = mutate(child2, rng, MUTATION_PROBABILITY)
            new_population.extend([child1, child2])

        population = new_population[:NPOP]
        fitness = evaluate_population(population)
        best_chromosome, best_fitness = update_best(
            population, fitness, best_chromosome, best_fitness
        )

        if verbose and generation % 25 == 0:
            finite_values = [value for value in fitness if math.isfinite(value)]
            generation_best = max(finite_values) if finite_values else float("-inf")
            print(f"Seed {ga_seed} | Generation {generation:3d} | Best objective {-generation_best:.2f}")

    if best_chromosome is None or not math.isfinite(best_fitness):
        raise RuntimeError(f"GA seed {ga_seed}: no valid solution was produced.")

    best_schedule, best_uncovered = decode_chromosome(best_chromosome[:])
    validation_errors = validate_schedule(best_schedule, best_uncovered)
    if validation_errors:
        raise RuntimeError(
            f"GA seed {ga_seed}: final solution failed validation: {validation_errors}"
        )

    components = objective_components(best_schedule, best_uncovered)
    runtime = time.perf_counter() - start_clock

    result = {
        "instance_seed": INSTANCE_SEED,
        "ga_seed": ga_seed,
        **components,
        "runtime_seconds": runtime,
        "feasible": True,
    }

    return result, best_schedule, best_uncovered, best_chromosome


In [89]:
# ============================================================
# FIVE INDEPENDENT GA REPLICATIONS
# ============================================================

GA_SEEDS = [0, 1, 2, 3, 4]

run_records = []
run_artifacts = {}

for ga_seed in GA_SEEDS:
    result, schedule, uncovered, chromosome = run_ga(ga_seed, verbose=False)
    run_records.append(result)
    run_artifacts[ga_seed] = {
        "schedule": schedule,
        "uncovered": uncovered,
        "chromosome": chromosome,
    }
    print(
        f"Seed {ga_seed}: ZT={result['total_objective']:.2f}, "
        f"Zd={result['delay_component']:.2f}, "
        f"Zt={result['travel_component']:.2f}, "
        f"Zu={result['coverage_component']:.2f}, "
        f"UP={result['uncovered_total']}, "
        f"time={result['runtime_seconds']:.2f}s"
    )

runs_df = pd.DataFrame(run_records).sort_values("ga_seed").reset_index(drop=True)

best_row_index = runs_df["total_objective"].idxmin()
best_row = runs_df.loc[best_row_index]
best_seed = int(best_row["ga_seed"])

summary_df = pd.DataFrame(
    {
        "number_of_runs": [len(runs_df)],
        "best_ga_seed": [best_seed],
        "best_total_objective": [runs_df["total_objective"].min()],
        "mean_total_objective": [runs_df["total_objective"].mean()],
        "sd_total_objective": [runs_df["total_objective"].std(ddof=1)],
        "cv_total_objective_percent": [
            100 * runs_df["total_objective"].std(ddof=1)
            / runs_df["total_objective"].mean()
        ],
        "mean_runtime_seconds": [runs_df["runtime_seconds"].mean()],
        "best_run_delay_component": [best_row["delay_component"]],
        "best_run_travel_component": [best_row["travel_component"]],
        "best_run_coverage_component": [best_row["coverage_component"]],
        "best_run_uncovered_total": [int(best_row["uncovered_total"])],
        "best_run_uncovered_advance": [int(best_row["uncovered_advance"])],
        "best_run_uncovered_dynamic": [int(best_row["uncovered_dynamic"])],
    }
)

runs_df.to_csv("ga_run_level_results.csv", index=False)
summary_df.to_csv("ga_summary_results.csv", index=False)

print("\nRun-level results")
display(runs_df.round(3))

print("\nAcross-run summary")
display(summary_df.round(3))


Seed 0: ZT=33449.33, Zd=2121.19, Zt=1328.14, Zu=30000.00, UP=8, time=85.57s
Seed 1: ZT=34807.67, Zd=1487.63, Zt=1320.04, Zu=32000.00, UP=9, time=90.03s
Seed 2: ZT=28714.90, Zd=1331.33, Zt=1383.57, Zu=26000.00, UP=8, time=87.84s
Seed 3: ZT=33596.54, Zd=1247.23, Zt=1349.31, Zu=31000.00, UP=9, time=90.96s
Seed 4: ZT=34212.92, Zd=1005.35, Zt=1207.57, Zu=32000.00, UP=9, time=65.48s

Run-level results


,instance_seed,ga_seed,total_objective,delay_component,travel_component,coverage_component,uncovered_priority,uncovered_total,uncovered_advance,uncovered_dynamic,runtime_seconds,feasible
0,42,0,33449.326,2121.186,1328.140,30000.0,30,8,5,3,85.568,True
1,42,1,34807.673,1487.630,1320.044,32000.0,32,9,4,5,90.031,True
2,42,2,28714.898,1331.326,1383.572,26000.0,26,8,4,4,87.835,True
3,42,3,33596.543,1247.233,1349.310,31000.0,31,9,3,6,90.962,True
4,42,4,34212.925,1005.354,1207.571,32000.0,32,9,2,7,65.479,True



Across-run summary


,number_of_runs,best_ga_seed,best_total_objective,mean_total_objective,sd_total_objective,cv_total_objective_percent,mean_runtime_seconds,best_run_delay_component,best_run_travel_component,best_run_coverage_component,best_run_uncovered_total,best_run_uncovered_advance,best_run_uncovered_dynamic
0,5,2,28714.898,32956.273,2431.52,7.378,83.975,1331.326,1383.572,26000.0,8,4,4
